# Tutorial 27: Fault Tolerance in LangGraph

A node that calls a payment API, a crawler, or another model will fail. LangGraph 1.2 lets you decide what happens next without wrapping every node in your own `try`. The mechanisms run in a fixed order:

1. A timeout can cancel one attempt.
2. A retry policy decides whether to try again.
3. An error handler runs only after retries are exhausted.

Graceful drain is separate. It stops the graph between supersteps and leaves a checkpoint you can resume.


## 1. Setup


In [1]:
import asyncio
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.errors import GraphDrained, NodeError
from langgraph.graph import END, START, StateGraph
from langgraph.runtime import RunControl
from langgraph.types import Command, RetryPolicy

print("Setup complete.")


Setup complete.


## 2. Retry a transient error

`RetryPolicy` retries a node when the exception matches `retry_on`. The handler below is not involved: the second attempt succeeds.

`MemorySaver` and `InMemorySaver` are the same class. This tutorial uses the current name.


In [2]:
class Status(TypedDict):
    status: str


attempts = {"n": 0}


def flaky(state: Status):
    attempts["n"] += 1
    if attempts["n"] < 2:
        raise ConnectionError("upstream blip")
    return {"status": "recovered"}


retry_graph = (
    StateGraph(Status)
    .add_node(
        "flaky",
        flaky,
        retry_policy=RetryPolicy(max_attempts=3, retry_on=ConnectionError),
    )
    .add_edge(START, "flaky")
    .compile()
)

print(retry_graph.invoke({"status": ""})["status"], "after", attempts["n"], "attempts")


recovered after 2 attempts


## 3. Compensate after the retries are gone

An `error_handler` receives the state and a `NodeError`. Return a `Command` to write state and route somewhere else. This is the saga pattern: the charge failed, so the graph records a compensation and still finishes.

`error.error` is the original exception. `interrupt()` does not go through this handler.


In [3]:
def charge(state: Status):
    raise RuntimeError("gateway down")


def compensate(state: Status, error: NodeError) -> Command:
    return Command(
        update={"status": f"compensated: {error.error}"},
        goto="finalize",
    )


def finalize(state: Status):
    return {"status": state["status"] + " / closed"}


saga = (
    StateGraph(Status)
    .add_node("charge", charge, error_handler=compensate)
    .add_node("finalize", finalize)
    .add_edge(START, "charge")
    .add_edge("finalize", END)
    .compile()
)

print(saga.invoke({"status": ""})["status"])


compensated: gateway down / closed


## 4. Cap one async attempt

`timeout=` on `add_node` limits a single attempt. It applies to async nodes. When the limit fires, LangGraph raises `NodeTimeoutError`, drops that attempt's writes, and then follows the retry policy. With no retry policy, the error handler runs immediately.

Pass a number of seconds, a `timedelta`, or `TimeoutPolicy(run_timeout=..., idle_timeout=...)` when you want a wall-clock cap and an idle cap.


In [4]:
async def sleepy(state: Status):
    await asyncio.sleep(0.3)
    return {"status": "finished"}


def on_timeout(state: Status, error: NodeError) -> Command:
    return Command(update={"status": type(error.error).__name__}, goto=END)


timeout_graph = (
    StateGraph(Status)
    .add_node("sleepy", sleepy, timeout=0.05, error_handler=on_timeout)
    .add_edge(START, "sleepy")
    .compile()
)

result = await timeout_graph.ainvoke({"status": ""})
print(result["status"])


NodeTimeoutError


## 5. Stop between steps and resume

`RunControl.request_drain()` does not cancel the node that is already running. The graph finishes the current superstep, saves a checkpoint, and raises `GraphDrained`. Call `invoke(None, config)` later with the same checkpointer and `thread_id` to continue.

Create a new `RunControl` for every run. Reusing one that has already drained keeps the run drained.


In [5]:
class Count(TypedDict):
    n: int


control = RunControl()


def first(state: Count):
    control.request_drain("deploy")
    return {"n": 1}


def second(state: Count):
    return {"n": state["n"] + 10}


drain_graph = (
    StateGraph(Count)
    .add_node(first)
    .add_node(second)
    .add_edge(START, "first")
    .add_edge("first", "second")
    .compile(checkpointer=InMemorySaver())
)
config = {"configurable": {"thread_id": "tutorial-27"}}

try:
    drain_graph.invoke({"n": 0}, config, control=control)
    raise AssertionError("the run should have stopped")
except GraphDrained as exc:
    print("stopped:", exc.reason)
    print("checkpoint:", drain_graph.get_state(config).values)

resumed = drain_graph.invoke(None, config)
print("resumed:", resumed["n"])


stopped: deploy
checkpoint: {'n': 1}
resumed: 11


## 6. Summary

- `retry_policy` repeats a failed attempt before anything else handles it.
- `error_handler` runs after those retries and can `Command(goto=...)` into a compensation node.
- `timeout` limits one async attempt. A timeout is a failure, so retries and the error handler still apply.
- `RunControl.request_drain()` stops after the current superstep and leaves a resumable checkpoint.

Use the same checkpointer you already used for memory (Tutorial 19) and human approval (Tutorial 14). Drain is how a process survives a deploy without throwing the thread away.
